# Fixed-dimensional predeclared lag ablation

This notebook tests whether temporal alignment itself explains forecast skill while
keeping sensor availability, feature count, forecast origins, targets, and model
definitions fixed. The only experimental factor is the Terneuzen lag: **24, 36, 42,
48, or 60 hours**. A deterministic **shuffled 42-hour negative control** preserves the
upstream feature distribution but breaks its alignment with the target.

Classical models are enabled by default. Neural execution is deliberately disabled by
`RUN_NEURAL = False`; changing that flag prepares equal-length, four-channel inputs for
LSTM, TCN, and their equal-weight ensemble, but no network is trained in the present
notebook state.

In [ ]:
from pathlib import Path
import math, os, sys, time, warnings

ROOT = Path.cwd().resolve()
if not (ROOT / "lag_analytics_workspace").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

from time_series_analysis.paper_experiment_utils import (
    FORECAST_STEPS,
    LAG_ABLATION_CONTROL,
    LAG_ABLATION_HOURS,
    build_classical_lag_ablation_datasets,
    build_neural_lag_ablation_datasets,
    evaluation_rows,
    load_prepared_observations,
    split_shape_table,
    validation_calibration,
)

SEED = 42
RUN_CLASSICAL = True
RUN_NEURAL = False  # Deliberately disabled for the present analysis pass.

CONDITION_ORDER = [f"lag_{hours}h" for hours in LAG_ABLATION_HOURS] + [LAG_ABLATION_CONTROL]
CONDITION_LABELS = {
    **{f"lag_{hours}h": f"{hours} h" for hours in LAG_ABLATION_HOURS},
    LAG_ABLATION_CONTROL: "Shuffled 42 h",
}
warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)

## 1. Matched data and negative-control design

Every tabular condition contains exactly four columns: the latest target value, its
one-hour change, its two-hour change, and one lagged Terneuzen value. The shuffled
control starts from the 42-hour condition and reassigns only the upstream feature among
complete examples, separately within training, validation, and test. Targets and local
predictors are never shuffled.

In [ ]:
data, coverage, unit_report = load_prepared_observations()
classical_datasets = build_classical_lag_ablation_datasets(
    data, lag_hours=LAG_ABLATION_HOURS, shuffle_seed=SEED
)
display(coverage)
display(split_shape_table(classical_datasets))

In [ ]:
def audit_matched_lag_design(datasets):
    reference_name = f"lag_{LAG_ABLATION_HOURS[0]}h"
    reference = datasets[reference_name]
    rows = []
    for condition in CONDITION_ORDER:
        dataset = datasets[condition]
        for split_name in ("train", "validation", "test"):
            split = dataset.splits[split_name]
            reference_split = reference.splits[split_name]
            assert split.X.shape == reference_split.X.shape
            assert split.X.shape[-1] == 4
            np.testing.assert_array_equal(split.forecast_times, reference_split.forecast_times)
            np.testing.assert_allclose(split.y_delta, reference_split.y_delta)
            np.testing.assert_allclose(split.baseline, reference_split.baseline)
            np.testing.assert_allclose(split.X[..., :3], reference_split.X[..., :3])
            rows.append({
                "Condition": CONDITION_LABELS[condition],
                "Split": split_name,
                "Input shape": str(split.X.shape),
                "Forecast origins": len(split.X),
                "First forecast": split.forecast_times.min(),
                "Last forecast": split.forecast_times.max(),
            })

    base = datasets["lag_42h"]
    control = datasets[LAG_ABLATION_CONTROL]
    for split_name in ("train", "validation", "test"):
        base_x = base.splits[split_name].X
        control_x = control.splits[split_name].X
        assert not np.allclose(base_x[..., -1], control_x[..., -1])
        np.testing.assert_allclose(
            np.sort(base_x[..., -1].ravel()),
            np.sort(control_x[..., -1].ravel()),
        )
    return pd.DataFrame(rows)

design_audit = audit_matched_lag_design(classical_datasets)
display(design_audit)

## 2. Classical models

Model definitions, chronological splits, 16-step residual target, August-only
calibration, and September evaluation are identical across all six conditions. The
regular RBF-SVR uses no PCA or feature reduction.

In [ ]:
def make_classical_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "XGBoost": XGBRegressor(
            n_estimators=250,
            learning_rate=0.04,
            max_depth=4,
            min_child_weight=5,
            subsample=0.80,
            colsample_bytree=0.65,
            reg_alpha=0.05,
            reg_lambda=5.0,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=SEED,
            n_jobs=min(8, os.cpu_count() or 1),
        ),
        "RBF-SVR": make_pipeline(
            StandardScaler(),
            MultiOutputRegressor(
                SVR(
                    kernel="rbf",
                    C=100.0,
                    epsilon=0.01,
                    cache_size=2048,
                ),
                n_jobs=min(4, os.cpu_count() or 1),
            ),
        ),
        "Shallow MLP": make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                learning_rate_init=1e-3,
                batch_size=256,
                max_iter=160,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=SEED,
            ),
        ),
    }

In [ ]:
def run_classical_lag_ablation(datasets):
    rows, horizon_frames, fit_rows = [], [], []
    predictions = {}
    for condition in CONDITION_ORDER:
        dataset = datasets[condition]
        train, validation, test = (
            dataset.splits[name] for name in ("train", "validation", "test")
        )
        scale = dataset.target_scale
        for model_name, model in make_classical_models().items():
            started = time.perf_counter()
            print(f"Fitting {model_name:12s} on {condition} ...", flush=True)
            model.fit(train.X, train.y_delta / scale)
            elapsed = time.perf_counter() - started

            validation_raw = np.asarray(model.predict(validation.X), dtype=np.float32) * scale
            weights = validation_calibration(validation_raw, validation.y_delta)
            validation_calibrated = validation_raw * weights
            validation_persistence_rmse = float(np.sqrt(np.mean(validation.y_delta ** 2)))
            validation_rmse = float(
                np.sqrt(np.mean((validation_calibrated - validation.y_delta) ** 2))
            )

            test_raw = np.asarray(model.predict(test.X), dtype=np.float32) * scale
            test_calibrated = test_raw * weights
            model_rows, horizon, prediction = evaluation_rows(
                model_name, condition, test, test_calibrated
            )
            rows.extend(model_rows)
            horizon_frames.append(horizon)
            predictions[(model_name, condition)] = prediction
            fit_rows.append({
                "Model": model_name,
                "Representation": condition,
                "Fit seconds": elapsed,
                "Validation RMSE": validation_rmse,
                "Validation persistence RMSE": validation_persistence_rmse,
                "Validation RMSE skill": 1 - validation_rmse / validation_persistence_rmse,
                "Mean calibration weight": float(weights.mean()),
            })
            print(
                f"  {elapsed:.1f}s; validation skill "
                f"{fit_rows[-1]['Validation RMSE skill']:.3f}"
            )
    return (
        pd.DataFrame(rows),
        pd.concat(horizon_frames, ignore_index=True),
        pd.DataFrame(fit_rows),
        predictions,
    )

if RUN_CLASSICAL:
    classical_results, classical_horizons, classical_fit, classical_predictions = (
        run_classical_lag_ablation(classical_datasets)
    )
else:
    classical_results = pd.DataFrame()
    classical_horizons = pd.DataFrame()
    classical_fit = pd.DataFrame()
    classical_predictions = {}

## 3. Optional neural lag sweep - disabled

The following code uses the same 42-hour sequence length and the same four channels for
every lag condition. Complete upstream sequences are shuffled among examples for the
negative control. `RUN_NEURAL` remains `False`, so running the notebook as supplied does
not import TensorFlow, construct networks, or train weights.

In [ ]:
def make_neural_builders():
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers

    tf.keras.utils.set_random_seed(SEED)
    L2 = 1e-5

    def build_lstm(input_shape):
        inputs = keras.Input(shape=input_shape, name="history")
        x = layers.LSTM(
            64, return_sequences=True, kernel_regularizer=regularizers.l2(L2)
        )(inputs)
        x = layers.Dropout(0.10)(x)
        x = layers.LSTM(32, kernel_regularizer=regularizers.l2(L2))(x)
        x = layers.LayerNormalization()(x)
        latest = layers.Lambda(lambda z: z[:, -1, :], name="latest_input")(inputs)
        x = layers.Concatenate()([x, latest])
        x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2))(x)
        x = layers.Dropout(0.10)(x)
        outputs = layers.Dense(
            FORECAST_STEPS, kernel_initializer="zeros", name="delta_forecast"
        )(x)
        return keras.Model(inputs, outputs, name="LSTM_residual_forecaster")

    def residual_tcn_block(x, filters, kernel_size, dilation):
        shortcut = x
        for conv_number in range(2):
            x = layers.Conv1D(
                filters,
                kernel_size,
                padding="causal",
                dilation_rate=dilation,
                kernel_regularizer=regularizers.l2(L2),
                name=f"d{dilation}_conv{conv_number + 1}",
            )(x)
            x = layers.LayerNormalization()(x)
            x = layers.Activation("relu")(x)
            x = layers.SpatialDropout1D(0.10)(x)
        if shortcut.shape[-1] != filters:
            shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
        return layers.Activation("relu")(layers.Add()([x, shortcut]))

    def build_tcn(input_shape):
        inputs = keras.Input(shape=input_shape, name="history")
        x = inputs
        for dilation in (1, 2, 4, 8, 16, 32):
            x = residual_tcn_block(x, filters=24, kernel_size=5, dilation=dilation)
        x = layers.Lambda(lambda z: z[:, -1, :], name="causal_last_state")(x)
        latest = layers.Lambda(lambda z: z[:, -1, :], name="latest_input")(inputs)
        x = layers.Concatenate()([x, latest])
        x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(L2))(x)
        x = layers.Dropout(0.10)(x)
        outputs = layers.Dense(
            FORECAST_STEPS, kernel_initializer="zeros", name="delta_forecast"
        )(x)
        return keras.Model(inputs, outputs, name="TCN_residual_forecaster")

    def compile_model(model):
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
            loss=keras.losses.Huber(delta=1.0),
            metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
        )
        return model

    return tf, keras, {"LSTM": build_lstm, "TCN": build_tcn}, compile_model


def run_neural_lag_ablation(datasets):
    tf, keras, builders, compile_model = make_neural_builders()
    epochs = {"LSTM": 40, "TCN": 45}
    batch_sizes = {"LSTM": 128, "TCN": 256}
    rows, horizon_frames, fit_rows = [], [], []
    predictions, validation_deltas, test_deltas = {}, {}, {}

    for condition in CONDITION_ORDER:
        dataset = datasets[condition]
        train, validation, test = (
            dataset.splits[name] for name in ("train", "validation", "test")
        )
        scale = dataset.target_scale
        for model_name, builder in builders.items():
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(SEED)
            model = compile_model(builder(train.X.shape[1:]))
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=7,
                    min_delta=1e-4,
                    restore_best_weights=True,
                    verbose=1,
                ),
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss",
                    factor=0.5,
                    patience=3,
                    min_lr=1e-5,
                    verbose=1,
                ),
            ]
            started = time.perf_counter()
            history = model.fit(
                train.X,
                train.y_delta / scale,
                validation_data=(validation.X, validation.y_delta / scale),
                epochs=epochs[model_name],
                batch_size=batch_sizes[model_name],
                shuffle=True,
                callbacks=callbacks,
                verbose=2,
            )
            elapsed = time.perf_counter() - started

            validation_raw = model.predict(validation.X, batch_size=512, verbose=0) * scale
            weights = validation_calibration(validation_raw, validation.y_delta)
            validation_calibrated = validation_raw * weights
            test_raw = model.predict(test.X, batch_size=512, verbose=0) * scale
            test_calibrated = test_raw * weights
            validation_deltas[(model_name, condition)] = validation_calibrated
            test_deltas[(model_name, condition)] = test_calibrated

            validation_persistence_rmse = float(np.sqrt(np.mean(validation.y_delta ** 2)))
            validation_rmse = float(
                np.sqrt(np.mean((validation_calibrated - validation.y_delta) ** 2))
            )
            model_rows, horizon, prediction = evaluation_rows(
                model_name, condition, test, test_calibrated
            )
            rows.extend(model_rows)
            horizon_frames.append(horizon)
            predictions[(model_name, condition)] = prediction
            fit_rows.append({
                "Model": model_name,
                "Representation": condition,
                "Parameters": model.count_params(),
                "Epochs": len(history.history["loss"]),
                "Fit seconds": elapsed,
                "Validation RMSE": validation_rmse,
                "Validation persistence RMSE": validation_persistence_rmse,
                "Validation RMSE skill": 1 - validation_rmse / validation_persistence_rmse,
                "Mean calibration weight": float(weights.mean()),
            })

        validation_ensemble = 0.5 * (
            validation_deltas[("LSTM", condition)]
            + validation_deltas[("TCN", condition)]
        )
        test_ensemble = 0.5 * (
            test_deltas[("LSTM", condition)] + test_deltas[("TCN", condition)]
        )
        validation_persistence_rmse = float(np.sqrt(np.mean(validation.y_delta ** 2)))
        validation_rmse = float(
            np.sqrt(np.mean((validation_ensemble - validation.y_delta) ** 2))
        )
        ensemble_rows, horizon, prediction = evaluation_rows(
            "LSTM+TCN ensemble", condition, test, test_ensemble
        )
        rows.extend(ensemble_rows)
        horizon_frames.append(horizon)
        predictions[("LSTM+TCN ensemble", condition)] = prediction
        fit_rows.append({
            "Model": "LSTM+TCN ensemble",
            "Representation": condition,
            "Parameters": np.nan,
            "Epochs": np.nan,
            "Fit seconds": 0.0,
            "Validation RMSE": validation_rmse,
            "Validation persistence RMSE": validation_persistence_rmse,
            "Validation RMSE skill": 1 - validation_rmse / validation_persistence_rmse,
            "Mean calibration weight": np.nan,
        })

    return (
        pd.DataFrame(rows),
        pd.concat(horizon_frames, ignore_index=True),
        pd.DataFrame(fit_rows),
        predictions,
    )

if RUN_NEURAL:
    neural_datasets = build_neural_lag_ablation_datasets(
        data, lag_hours=LAG_ABLATION_HOURS, sequence_hours=42, shuffle_seed=SEED
    )
    neural_results, neural_horizons, neural_fit, neural_predictions = (
        run_neural_lag_ablation(neural_datasets)
    )
else:
    neural_datasets = {}
    neural_results = pd.DataFrame()
    neural_horizons = pd.DataFrame()
    neural_fit = pd.DataFrame()
    neural_predictions = {}
    print("Neural lag sweep skipped: RUN_NEURAL is False.")

## 4. Fixed-lag comparison and shuffled-control contrasts

Candidate lags are compared with the shuffled control using paired test origins. Lag
ranking must use August validation RMSE; September is used only to report the chosen
configurations and the predeclared full sweep.

In [ ]:
result_frames = [frame for frame in (classical_results, neural_results) if not frame.empty]
horizon_frames = [frame for frame in (classical_horizons, neural_horizons) if not frame.empty]
fit_frames = [frame for frame in (classical_fit, neural_fit) if not frame.empty]
if not result_frames:
    raise RuntimeError("No experiment results are available. Enable at least one model family.")

results = pd.concat(result_frames, ignore_index=True)
horizon_results = pd.concat(horizon_frames, ignore_index=True)
fit_summary = pd.concat(fit_frames, ignore_index=True)
results["Condition"] = results["Representation"].map(CONDITION_LABELS)
results["Condition"] = pd.Categorical(
    results["Condition"],
    categories=[CONDITION_LABELS[name] for name in CONDITION_ORDER],
    ordered=True,
)

summary = results.pivot_table(
    index=["Model", "Condition"],
    columns="Scope",
    values=["RMSE", "R2", "RMSE skill"],
    observed=False,
).sort_index()
display(summary.round(4))

control = results[results["Representation"] == LAG_ABLATION_CONTROL][
    ["Model", "Scope", "RMSE", "RMSE skill"]
].rename(columns={
    "RMSE": "Shuffled-control RMSE",
    "RMSE skill": "Shuffled-control skill",
})
control_comparison = results[
    results["Representation"] != LAG_ABLATION_CONTROL
].merge(control, on=["Model", "Scope"], validate="many_to_one")
control_comparison["RMSE reduction vs shuffled"] = (
    control_comparison["Shuffled-control RMSE"] - control_comparison["RMSE"]
)
control_comparison["Skill gain vs shuffled"] = (
    control_comparison["RMSE skill"] - control_comparison["Shuffled-control skill"]
)
display(control_comparison.sort_values(
    ["Model", "Scope", "RMSE reduction vs shuffled"],
    ascending=[True, True, False],
).round(4))

## 5. Visualization: lag response, negative control, and model-by-lag heatmap

In [ ]:
def plot_lag_response(results_frame):
    real_names = [f"lag_{hours}h" for hours in LAG_ABLATION_HOURS]
    control_x = max(LAG_ABLATION_HOURS) + 8
    panels = [
        ("all horizons", "RMSE skill", "Overall skill vs persistence"),
        ("4-hour endpoint", "RMSE skill", "Four-hour endpoint skill"),
        ("largest 10% changes", "RMSE skill", "Largest-change skill"),
        ("all horizons", "RMSE", "Overall RMSE"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
    colors = plt.cm.tab10(np.linspace(0, 1, results_frame["Model"].nunique()))
    color_map = dict(zip(sorted(results_frame["Model"].unique()), colors))

    for ax, (scope, metric, title) in zip(axes.ravel(), panels):
        scoped = results_frame[results_frame["Scope"] == scope]
        for model_name, group in scoped.groupby("Model"):
            real = group.set_index("Representation").reindex(real_names)
            ax.plot(
                LAG_ABLATION_HOURS,
                real[metric],
                marker="o",
                linewidth=1.8,
                label=model_name,
                color=color_map[model_name],
            )
            control_row = group[group["Representation"] == LAG_ABLATION_CONTROL]
            if not control_row.empty:
                ax.scatter(
                    [control_x],
                    [control_row.iloc[0][metric]],
                    marker="X",
                    s=85,
                    color=color_map[model_name],
                    edgecolor="black",
                    linewidth=0.5,
                    zorder=4,
                )
        if metric == "RMSE skill":
            ax.axhline(0, color="black", linewidth=1, linestyle="--")
            ax.set_ylabel("RMSE skill (positive beats persistence)")
        else:
            persistence_rmse = float(scoped["Persistence RMSE"].iloc[0])
            ax.axhline(
                persistence_rmse,
                color="black",
                linewidth=1,
                linestyle="--",
                label="Persistence",
            )
            ax.set_ylabel("RMSE (mS/cm)")
        ax.set_title(title)
        ax.set_xticks([*LAG_ABLATION_HOURS, control_x])
        ax.set_xticklabels([*[f"{hours} h" for hours in LAG_ABLATION_HOURS], "Shuffled\n42 h"])
        ax.set_xlabel("Terneuzen feature alignment")
        ax.grid(alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncol=min(4, len(labels)))
    plt.show()


def plot_skill_heatmap(results_frame, scope="all horizons"):
    labels = [CONDITION_LABELS[name] for name in CONDITION_ORDER]
    matrix = (
        results_frame[results_frame["Scope"] == scope]
        .pivot(index="Model", columns="Condition", values="RMSE skill")
        .reindex(columns=labels)
    )
    values = matrix.to_numpy(dtype=float)
    limit = max(0.01, float(np.nanmax(np.abs(values))))
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.65 * len(matrix))))
    image = ax.imshow(values, aspect="auto", cmap="RdYlGn", vmin=-limit, vmax=limit)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title(f"{scope.title()} RMSE skill by model and lag condition")
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            if np.isfinite(value):
                ax.text(column, row, f"{100 * value:.1f}%", ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=ax, label="RMSE skill")
    plt.tight_layout()
    plt.show()


plot_lag_response(results)
plot_skill_heatmap(results)

## 6. Validation-selected lag and horizon diagnostics

For each model, the preferred physical lag is selected from the five real lag conditions
using August only. Its September skill curve is then displayed beside the shuffled
42-hour control. The control is never eligible for selection.

In [ ]:
real_conditions = [f"lag_{hours}h" for hours in LAG_ABLATION_HOURS]
selected = (
    fit_summary[fit_summary["Representation"].isin(real_conditions)]
    .sort_values(["Model", "Validation RMSE"])
    .groupby("Model", as_index=False)
    .first()
)
selected["Selected condition"] = selected["Representation"].map(CONDITION_LABELS)
display(selected[[
    "Model", "Selected condition", "Validation RMSE",
    "Validation RMSE skill", "Fit seconds"
]].round(4))

models = list(selected["Model"])
columns = 2
rows = math.ceil(len(models) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(14, 4.2 * rows), squeeze=False)
for ax, model_name in zip(axes.ravel(), models):
    chosen = selected.loc[selected["Model"] == model_name, "Representation"].iloc[0]
    for condition, style in ((chosen, "-"), (LAG_ABLATION_CONTROL, "--")):
        group = horizon_results[
            (horizon_results["Model"] == model_name)
            & (horizon_results["Representation"] == condition)
        ]
        ax.plot(
            group["Lead minutes"],
            group["RMSE skill"],
            marker="o",
            linestyle=style,
            label=CONDITION_LABELS[condition],
        )
    ax.axhline(0, color="black", linewidth=1)
    ax.set(
        title=model_name,
        xlabel="Lead time (minutes)",
        ylabel="RMSE skill",
    )
    ax.grid(alpha=0.25)
    ax.legend()
for ax in axes.ravel()[len(models):]:
    ax.set_visible(False)
fig.suptitle("September horizon skill: August-selected lag vs shuffled control", y=1.01)
plt.tight_layout()
plt.show()

## 7. Reproducible reporting summary

In [ ]:
for _, chosen in selected.iterrows():
    model_name = chosen["Model"]
    condition = chosen["Representation"]
    test_row = results[
        (results["Model"] == model_name)
        & (results["Representation"] == condition)
        & (results["Scope"] == "all horizons")
    ].iloc[0]
    control_row = results[
        (results["Model"] == model_name)
        & (results["Representation"] == LAG_ABLATION_CONTROL)
        & (results["Scope"] == "all horizons")
    ].iloc[0]
    print(
        f"{model_name}: August selected {CONDITION_LABELS[condition]}; "
        f"September RMSE={test_row['RMSE']:.5f}, "
        f"skill={test_row['RMSE skill']:.3f}; "
        f"RMSE reduction vs shuffled control="
        f"{control_row['RMSE'] - test_row['RMSE']:.5f}."
    )

print(
    "\nInterpretation rule: a preferred lag should improve upon both persistence and "
    "the shuffled control, remain stable across forecast horizons, and be selected "
    "using validation data rather than September test performance."
)